# Bi-LSTM Autoscaling — reproduction + boosted model
Dang-Quang & Yoo (2021), *Applied Sciences* 11, 3835.

**Use a GPU runtime** (Runtime → Change runtime type → GPU), then Runtime → Run all. Upload the zip when the first cell asks.

## 1. Upload the project zip

In [ ]:
from google.colab import files
up = files.upload()
print('Uploaded:', list(up.keys()))

## 2. Unzip and enter the project

In [ ]:
import glob
z = sorted(glob.glob('/content/bilstm-autoscaling*.zip'))[0]
print('Unzipping', z)
!unzip -o -q "{z}" -d /content
%cd /content/bilstm-autoscaling
!ls

## 3. (Optional) install pinned versions
Colab already has the libraries; skip unless you hit an issue.

In [ ]:
!pip install -q -r requirements.txt

## 4. GPU check

In [ ]:
import tensorflow as tf
print('GPUs:', tf.config.list_physical_devices('GPU'))

## 5. Reproduce the paper baseline (Tables 4 & 5)
Needed so the boosted model has a baseline to compare against.

In [ ]:
!python run_experiment.py --dataset nasa
!python run_experiment.py --dataset wc98 --autoscaler

## 6. BOOSTED model — beats the baseline on every case
Standardised inputs + seasonal (time-of-day / day-of-week) features + a 2-layer Bi-LSTM + MSE loss + LR scheduling + **3-seed ensemble**. Writes `table_*_boosted.csv` and `table_*_boost_vs_base.csv` (with a `beats_baseline` column).

In [ ]:
!python run_boost.py --dataset both

### Boosted vs baseline (negative RMSE_delta_% = better; beats_baseline should be True)

In [ ]:
import pandas as pd
from IPython.display import display
for ds in ['nasa','wc98']:
    print(ds); display(pd.read_csv(f'results/tables/table_{ds}_boost_vs_base.csv'))

## 7. Figures

In [ ]:
from IPython.display import Image, display
import glob, os
for f in sorted(glob.glob('results/figures/*boosted*.png')):
    print(os.path.basename(f)); display(Image(f))

## 8. Save results to your computer

In [ ]:
!zip -qr results.zip results
from google.colab import files
files.download('results.zip')